# 🧠 CerebroSpheroid V4: Cloud TPU Training with JAX in Google Colab
### 127M Parameters • Pure JAX / XLA Native • Automatic Google Drive PyTorch (.pt) Sync

This notebook trains the **CerebroSpheroid V4** architecture natively on Google Colab Cloud TPUs (TPU v2 / v3 / v4 / v5e) using JAX and XLA, with **automatic streaming of PyTorch (`.pt`) checkpoints directly to your Google Drive (`/MyDrive/cerebro_checkpoints`)**.

### Key Advantages:
1. **Automatic Google Drive Auto-Sync**: Checkpoints are saved natively as PyTorch `state_dict` (`.pt`) and mirrored directly into your Google Drive every interval and on new best loss.
2. **Zero PyTorch/XLA Recompilation Overhead**: The entire multi-step deliberation and zero-feedback autoregressive emission loop is compiled into a single fused XLA kernel via `jax.lax.scan`.
3. **Native TPU Matrix Performance**: Directly executes on the TPU Systolic Matrix Units (MXUs) in `bfloat16`.

---
### ⚡ Colab Settings:
1. Click **Runtime** -> **Change runtime type**
2. Hardware accelerator: Select **TPU v2** (or **TPU v4** / **TPU v5e** if Colab Pro)
3. Click **Save** and run the cells below in order.

## Step 1: Verify Google Colab Cloud TPU & JAX Environment

In [ ]:
import jax
import jax.numpy as jnp

print(f"JAX Version      : {jax.__version__}")
print(f"JAX Default Dev  : {jax.default_backend().upper()}")
devices = jax.devices()
print(f"Detected Devices : {devices}")
print(f"Device type      : {devices[0].device_kind}")

is_tpu = any("TPU" in d.device_kind.upper() for d in devices)
if is_tpu:
    print(f"\n✅ Successfully connected to Cloud TPU! ({len(devices)} cores active)")
else:
    print("\n⚠️ Running on CPU/GPU. For maximum speed, go to Runtime -> Change runtime type -> TPU.")


## Step 2: Mount Google Drive for Automatic Checkpoint Persistence
All model checkpoints (`.pt`) and metrics (`.json`) will be saved directly into `/content/drive/MyDrive/cerebro_checkpoints` so your training progress is permanently preserved.

In [ ]:
import os, shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Destination directory in your Google Drive
DRIVE_CKPT_DIR = "/content/drive/MyDrive/cerebro_checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f"✅ Google Drive successfully mounted!")
print(f"📂 Checkpoints destination: {DRIVE_CKPT_DIR}")

def save_to_drive(local_path: str):
    """Copies checkpoint to Google Drive and verifies file sync."""
    filename = os.path.basename(local_path)
    dest_path = os.path.join(DRIVE_CKPT_DIR, filename)
    shutil.copy2(local_path, dest_path)
    # Also copy companion JSON metadata if present
    json_local = local_path.replace(".pt", ".json")
    if os.path.exists(json_local) and json_local != local_path:
        shutil.copy2(json_local, os.path.join(DRIVE_CKPT_DIR, os.path.basename(json_local)))
    size_mb = os.path.getsize(dest_path) / (1024**2)
    print(f"  💾 [Google Drive] Saved {filename} ({size_mb:.1f} MB) -> {DRIVE_CKPT_DIR}/")


## Step 3: Install Supporting Libraries

In [ ]:
!pip install -q transformers datasets sentencepiece optax torch
import os
if not os.path.exists('reasoning_agent'):
    !git clone https://github.com/AjitKumar-git/reasoning_agent.git
%cd reasoning_agent
!git pull
print('✅ Dependencies installed and repository synchronized successfully!')


## Step 4: CerebroSpheroid V4 Architecture Definition (Pure JAX)

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Any
import math
import numpy as np
import torch, json
from jax import lax

DEFAULT_RMS_EPS = 1e-6

@dataclass
class ModelConfig:
    vocab_size: int = 32000          # LLaMA 32k Byte-Fallback BPE
    surface_dim: int = 640           # Spheroid boundary dimension
    num_shells: int = 16             # Concentric spherical laminae
    radial_dim: int = 256            # Radial wave channel depth
    num_shortcut_tracts: int = 40    # Axonal shortcut tracts
    max_phase_steps: int = 1024      # Premotor phase clock
    thinking_cycles: int = 3         # Subconscious deliberation steps
    use_shell_tap: bool = True       # Multi-shell efferent tap
    use_plan: bool = True            # Recurrent premotor plan

def rms_norm(x: jnp.ndarray, weight: jnp.ndarray, eps: float = DEFAULT_RMS_EPS) -> jnp.ndarray:
    rms = jnp.sqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + eps)
    return (x / rms) * weight

def precompute_axonal_indices(num_shells: int, num_shortcut_tracts: int) -> Tuple[jnp.ndarray, jnp.ndarray]:
    src = jnp.arange(num_shortcut_tracts) % num_shells
    dst = num_shells - 1 - (jnp.arange(num_shortcut_tracts) * 3) % num_shells
    return src.astype(jnp.int32), dst.astype(jnp.int32)

def init_v4_params(config: ModelConfig, key: jax.Array) -> Tuple[Dict[str, jnp.ndarray], Tuple[jnp.ndarray, jnp.ndarray]]:
    keys = jax.random.split(key, 20)
    vocab, s_dim, shells, r_dim = config.vocab_size, config.surface_dim, config.num_shells, config.radial_dim
    n_tracts, fused_dim = config.num_shortcut_tracts, s_dim * 5
    src_idx, dst_idx = precompute_axonal_indices(shells, n_tracts)

    p: Dict[str, jnp.ndarray] = {
        "core.intra_shell": jax.random.normal(keys[0], (shells, s_dim, s_dim)) * 0.015,
        "core.reentrant_field": jax.random.normal(keys[1], (shells, shells, r_dim, r_dim)) * 0.008,
        "core.shortcut_tracts": jax.random.normal(keys[2], (n_tracts, s_dim, s_dim)) * 0.01,
        "core.radial_proj_in.weight": jax.random.normal(keys[3], (r_dim, s_dim)) * (1.0 / math.sqrt(s_dim)),
        "core.radial_proj_out.weight": jax.random.normal(keys[4], (s_dim, r_dim)) * (1.0 / math.sqrt(r_dim)),
        "core.norm.weight": jnp.ones(s_dim, dtype=jnp.float32),
        "core.flow_norm.weight": jnp.ones(s_dim, dtype=jnp.float32),
        "core.log_write": jnp.full((shells,), math.log(math.expm1(0.35)), dtype=jnp.float32),
        "core.log_leak": jnp.full((shells,), math.log(0.35 / 0.65), dtype=jnp.float32),

        # Sensory Front-End
        "sensory.embedding.weight": jax.random.normal(keys[5], (vocab, s_dim)) * 0.02,
        "sensory.fc1_gate.weight": jax.random.normal(keys[6], (s_dim * 2, s_dim)) * (1.0 / math.sqrt(s_dim)),
        "sensory.fc1_val.weight": jax.random.normal(keys[7], (s_dim * 2, s_dim)) * (1.0 / math.sqrt(s_dim)),
        "sensory.norm1.weight": jnp.ones(s_dim * 2, dtype=jnp.float32),
        "sensory.fc2.weight": jax.random.normal(keys[8], (s_dim, s_dim * 2)) * (1.0 / math.sqrt(s_dim * 2)),
        "sensory.norm_out.weight": jnp.ones(s_dim, dtype=jnp.float32),

        # Motor Output Decoder
        "motor.phase_embed.weight": jax.random.normal(keys[9], (config.max_phase_steps, s_dim)) * 0.02,
        "motor.q_proj.weight": jax.random.normal(keys[10], (s_dim, s_dim)) * (1.0 / math.sqrt(s_dim)),
        "motor.k_proj.weight": jax.random.normal(keys[11], (s_dim, s_dim)) * (1.0 / math.sqrt(s_dim)),
        "motor.v_proj.weight": jax.random.normal(keys[12], (s_dim, s_dim)) * (1.0 / math.sqrt(s_dim)),
        "motor.resonance_norm.weight": jnp.ones(s_dim, dtype=jnp.float32),
        "motor.shell_proj.weight": jax.random.normal(keys[13], (s_dim, s_dim)) * (1.0 / math.sqrt(s_dim)),
        "motor.shell_score": jnp.zeros((shells,), dtype=jnp.float32),
        "motor.shell_scale": jnp.zeros((1,), dtype=jnp.float32),
        "motor.plan_gate.weight": jax.random.normal(keys[14], (s_dim, s_dim * 3)) * (1.0 / math.sqrt(s_dim * 3)),
        "motor.plan_gate.bias": jnp.zeros((s_dim,), dtype=jnp.float32),
        "motor.plan_update.weight": jax.random.normal(keys[15], (s_dim, s_dim * 3)) * (1.0 / math.sqrt(s_dim * 3)),
        "motor.plan_write": jnp.zeros((1,), dtype=jnp.float32),
        "motor.plan_decay": jnp.array(0.90, dtype=jnp.float32),
        "motor.fused_norm.weight": jnp.ones(fused_dim, dtype=jnp.float32),
        "motor.fc_gate.weight": jax.random.normal(keys[16], (s_dim * 2, fused_dim)) * (1.0 / math.sqrt(fused_dim)),
        "motor.fc_val.weight": jax.random.normal(keys[17], (s_dim * 2, fused_dim)) * (1.0 / math.sqrt(fused_dim)),
        "motor.norm_mid.weight": jnp.ones(s_dim * 2, dtype=jnp.float32),
        "motor.head.weight": jax.random.normal(keys[18], (vocab, s_dim * 2)) * (1.0 / math.sqrt(s_dim * 2)),
        "motor.temperature_scale": jnp.array(1.0, dtype=jnp.float32),
        "motor.refractory_strength": jnp.array(1.2, dtype=jnp.float32),
        "motor.refractory_decay": jnp.array(0.75, dtype=jnp.float32),
        "motor.efference_proj.weight": jax.random.normal(keys[19], (s_dim, s_dim * 2)) * 0.01,
    }
    buffers = (src_idx, dst_idx)
    return p, buffers

def encode_sensory(params: Dict[str, jnp.ndarray], token_ids: jnp.ndarray, config: ModelConfig) -> Tuple[jnp.ndarray, Tuple[jnp.ndarray, jnp.ndarray]]:
    x = params["sensory.embedding.weight"][token_ids]
    gate = jax.nn.silu(jnp.matmul(x, params["sensory.fc1_gate.weight"].T))
    val = jnp.matmul(x, params["sensory.fc1_val.weight"].T)
    h = rms_norm(gate * val, params["sensory.norm1.weight"])
    flux = jnp.matmul(h, params["sensory.fc2.weight"].T) + x
    sensory_memory = rms_norm(flux, params["sensory.norm_out.weight"])
    keys = jnp.matmul(sensory_memory, params["motor.k_proj.weight"].T)
    values = jnp.matmul(sensory_memory, params["motor.v_proj.weight"].T)
    return sensory_memory, (keys, values)

def deliberate_step(params: Dict[str, jnp.ndarray], state: jnp.ndarray, buffers: Tuple[jnp.ndarray, jnp.ndarray], config: ModelConfig) -> jnp.ndarray:
    B, K, D = state.shape
    src, dst = buffers
    intra = jnp.einsum('bki,kij->bkj', state, params["core.intra_shell"])
    state_src = state[:, src, :]
    tracts_out = jnp.einsum('bti,tij->btj', state_src, params["core.shortcut_tracts"])
    tract_acc = jnp.zeros_like(state).at[:, dst, :].add(tracts_out)
    rad_in = jnp.matmul(state, params["core.radial_proj_in.weight"].T)
    reentrant = jnp.einsum('bki,kjio->bjo', rad_in, params["core.reentrant_field"])
    reentrant = jnp.matmul(jax.nn.silu(reentrant), params["core.radial_proj_out.weight"].T)
    flow = rms_norm(intra + tract_acc + reentrant, params["core.flow_norm.weight"])
    leak = jax.nn.sigmoid(params["core.log_leak"])[None, :, None]
    write = jax.nn.softplus(params["core.log_write"])[None, :, None]
    raw = (1.0 - leak) * state + write * jnp.tanh(flow)
    return rms_norm(raw, params["core.norm.weight"])

def emit_step(
    params: Dict[str, jnp.ndarray],
    state: jnp.ndarray,
    sensory_kv: Tuple[jnp.ndarray, jnp.ndarray],
    plan: jnp.ndarray,
    refractory_trace: Optional[jnp.ndarray],
    step_idx: int,
    buffers: Tuple[jnp.ndarray, jnp.ndarray],
    config: ModelConfig,
) -> Tuple[jnp.ndarray, jnp.ndarray, Optional[jnp.ndarray], jnp.ndarray]:
    B, K, D = state.shape
    surface, core = state[:, -1, :], state[:, 0, :]
    phase_idx = jnp.clip(step_idx, 0, config.max_phase_steps - 1)
    phase = params["motor.phase_embed.weight"][phase_idx]
    motor_surface = surface + phase

    keys, values = sensory_kv
    query = jnp.matmul(motor_surface, params["motor.q_proj.weight"].T)[:, None, :]
    scores = jnp.matmul(query, jnp.swapaxes(keys, -1, -2)) / math.sqrt(D)
    attn_weights = jax.nn.softmax(scores, axis=-1)
    resonance = rms_norm(jnp.squeeze(jnp.matmul(attn_weights, values), axis=1), params["motor.resonance_norm.weight"])

    shell_weights = jax.nn.softmax(params["motor.shell_score"], axis=0)[None, :, None]
    shell_proj = jnp.matmul(state, params["motor.shell_proj.weight"].T)
    shell_pool = jnp.sum(shell_weights * shell_proj, axis=1) * params["motor.shell_scale"]

    fused = rms_norm(
        jnp.concatenate([motor_surface, core, resonance, shell_pool, plan], axis=-1),
        params["motor.fused_norm.weight"],
    )
    gate = jax.nn.silu(jnp.matmul(fused, params["motor.fc_gate.weight"].T))
    val = jnp.matmul(fused, params["motor.fc_val.weight"].T)
    h = rms_norm(gate * val, params["motor.norm_mid.weight"])

    phase_batch = jnp.repeat(phase[None, :], B, axis=0)
    plan_input = jnp.concatenate([h, phase_batch], axis=-1)
    p_gate = jax.nn.sigmoid(jnp.matmul(plan_input, params["motor.plan_gate.weight"].T) + params["motor.plan_gate.bias"])
    next_plan = params["motor.plan_decay"] * plan + params["motor.plan_write"] * p_gate * jnp.matmul(plan_input, params["motor.plan_update.weight"].T)

    temp = jnp.clip(params["motor.temperature_scale"], 0.1, 5.0)
    raw_logits = jnp.matmul(h, params["motor.head.weight"].T) / temp
    efference = jnp.matmul(h, params["motor.efference_proj.weight"].T)

    if refractory_trace is not None:
        alpha = jnp.clip(params["motor.refractory_strength"], 0.1, 5.0)
        adapted_logits = raw_logits - alpha * refractory_trace
        decay = jax.nn.sigmoid(params["motor.refractory_decay"])
        current_prob = lax.stop_gradient(jax.nn.softmax(raw_logits, axis=-1))
        next_trace = lax.stop_gradient(decay * refractory_trace + (1.0 - decay) * current_prob)
    else:
        adapted_logits, next_trace = raw_logits, None

    state_eff = state.at[:, 0, :].add(efference)
    next_state = deliberate_step(params, state_eff, buffers, config)
    return adapted_logits, next_state, next_trace, next_plan

def training_rollout(
    params: Dict[str, jnp.ndarray],
    prompts: jnp.ndarray,
    target_len: int,
    buffers: Tuple[jnp.ndarray, jnp.ndarray],
    config: ModelConfig
) -> jnp.ndarray:
    B, L = prompts.shape
    sensory_mem, sensory_kv = encode_sensory(params, prompts, config)
    init_state = jnp.zeros((B, config.num_shells, config.surface_dim), dtype=jnp.float32)
    state = init_state.at[:, -1, :].set(jnp.mean(sensory_mem, axis=1))

    # Deliberate
    def _delib(s, _): return deliberate_step(params, s, buffers, config), None
    state, _ = lax.scan(_delib, state, None, length=config.thinking_cycles)

    # Emit
    init_plan = jnp.zeros((B, config.surface_dim), dtype=jnp.float32)
    init_refractory = jnp.zeros((B, config.vocab_size), dtype=jnp.float32)

    def _emit(carry, step_idx):
        cur_state, cur_trace, cur_plan = carry
        logits, nxt_s, nxt_t, nxt_p = emit_step(params, cur_state, sensory_kv, cur_plan, cur_trace, step_idx, buffers, config)
        return (nxt_s, nxt_t, nxt_p), logits

    _, all_logits = lax.scan(_emit, (state, init_refractory, init_plan), jnp.arange(target_len))
    return jnp.swapaxes(all_logits, 0, 1)

# =====================================================================
# 💾 Save JAX Parameters to PyTorch (.pt) Format
# =====================================================================
def save_jax_to_torch_pt(params: Dict[str, jnp.ndarray], buffers: Tuple[jnp.ndarray, jnp.ndarray], save_path: str, meta: Optional[dict] = None):
    """Converts JAX PyTree into standard PyTorch state_dict and saves to .pt format."""
    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    torch_state: Dict[str, torch.Tensor] = {}
    for k, v in params.items():
        torch_state[k] = torch.from_numpy(np.array(v))
    torch_state["core.src_indices"] = torch.from_numpy(np.array(buffers[0]))
    torch_state["core.dst_indices"] = torch.from_numpy(np.array(buffers[1]))
    torch.save(torch_state, save_path)
    if meta:
        meta_path = save_path.replace(".pt", ".json")
        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)
    print(f"  💾 Created local PyTorch checkpoint: {save_path}")

print("✅ CerebroSpheroid V4 Architecture and PyTorch Converter ready!")


## Step 5: JIT-Compiled Training Step (`@jax.jit`)

In [ ]:
from transformers import AutoTokenizer

try:
    tokenizer = AutoTokenizer.from_pretrained("huggyllama/llama-7b")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained("gpt2")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
pad_id = tokenizer.pad_token_id

config = ModelConfig(vocab_size=tokenizer.vocab_size)
key = jax.random.PRNGKey(1337)
key, init_k = jax.random.split(key)

# Decoupled float params from integer routing buffers
params, buffers = init_v4_params(config, init_k)
param_count = sum(p.size for p in params.values() if jnp.issubdtype(p.dtype, jnp.inexact))
print(f"Model scale: {param_count:,} parameters ({param_count * 2 / (1024**2):.1f} MB in bf16)")

# Pure JAX AdamW state
m_state = jax.tree_util.tree_map(jnp.zeros_like, params)
v_state = jax.tree_util.tree_map(jnp.zeros_like, params)
opt_state = {"step": jnp.array(0, dtype=jnp.int32), "m": m_state, "v": v_state}

# JIT-compiled with allow_int=True (handles discrete tokens without type errors)
@jax.jit
def train_step(p, opt_s, q_batch, r_batch, lr):
    target_len = r_batch.shape[1]
    def _loss_fn(param_dict):
        logits = training_rollout(param_dict, q_batch, target_len, buffers, config)
        log_probs = jax.nn.log_softmax(logits, axis=-1)
        targets_onehot = jax.nn.one_hot(r_batch, config.vocab_size)
        token_loss = -jnp.sum(targets_onehot * log_probs, axis=-1)
        mask = (r_batch != pad_id).astype(jnp.float32)
        return jnp.sum(token_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)

    loss_val, grads = jax.value_and_grad(_loss_fn, allow_int=True)(p)
    
    # Clip grads
    grad_norm = jnp.sqrt(sum(jnp.sum(jnp.square(g)) for g in grads.values() if jnp.issubdtype(g.dtype, jnp.inexact)))
    clip_scale = jnp.minimum(1.0, 1.0 / (grad_norm + 1e-6))
    grads = jax.tree_util.tree_map(lambda g: g * clip_scale if jnp.issubdtype(g.dtype, jnp.inexact) else g, grads)

    # AdamW update
    step = opt_s["step"] + 1
    def _leaf_up(param, grad, m, v):
        if not jnp.issubdtype(param.dtype, jnp.inexact):
            return param, m, v
        m_new = 0.9 * m + 0.1 * grad
        v_new = 0.999 * v + 0.001 * jnp.square(grad)
        m_hat = m_new / (1.0 - jnp.power(0.9, step))
        v_hat = v_new / (1.0 - jnp.power(0.999, step))
        param_new = param - lr * (m_hat / (jnp.sqrt(v_hat) + 1e-8) + 0.01 * param)
        return param_new, m_new, v_new

    flat_p, treedef = jax.tree_util.tree_flatten(p)
    flat_g, _ = jax.tree_util.tree_flatten(grads)
    flat_m, _ = jax.tree_util.tree_flatten(opt_s["m"])
    flat_v, _ = jax.tree_util.tree_flatten(opt_s["v"])
    updated = [_leaf_up(param, g, m, v) for param, g, m, v in zip(flat_p, flat_g, flat_m, flat_v)]

    new_p = jax.tree_util.tree_unflatten(treedef, [u[0] for u in updated])
    new_m = jax.tree_util.tree_unflatten(treedef, [u[1] for u in updated])
    new_v = jax.tree_util.tree_unflatten(treedef, [u[2] for u in updated])
    return new_p, {"step": step, "m": new_m, "v": new_v}, loss_val

print("✅ JIT compilation ready for Cloud TPU MXUs!")


## Step 6: Launch Cloud TPU Training with Automatic Google Drive (.pt) Sync
Checkpoints are saved locally and immediately mirrored into `/content/drive/MyDrive/cerebro_checkpoints/` every interval and whenever a new best loss is achieved!

In [ ]:
# 🚀 Launch CerebroSpheroid V4 JAX Training Pipeline
# Features:
# 1. Automatic checkpoint discovery & auto-resume from Google Drive
# 2. SpheroidOptimizer (Newton-Schulz NS5 polar updates + persistent contractive spectral clamp)
# 3. Elastic Sliding Context Window (balances prompt vs response up to 1536 total sequence budget)
# 4. Asynchronous ThreadPool Google Drive syncing (0 ms TPU pause during checkpoint writes)
# 5. EMA-gated best checkpoint evaluation (eliminates short-batch low loss artifacts)

!python3 training/train_v4_jax.py \n    --drive_dir /content/drive/MyDrive/cerebro_checkpoints \n    --elastic_window \n    --total_window 1536 \n    --batch_size 16 \n    --lr 0.002 \n    --max_steps 50000 \n    --save_every 100 \n    --log_every 10


## Step 7: Export Final PyTorch (.pt) Checkpoint to Google Drive

In [ ]:
final_path = "cerebro_spheroid_v4_jax_final.pt"
save_jax_to_torch_pt(params, buffers, final_path, meta={"total_steps": TRAIN_STEPS, "best_loss": best_loss})
save_to_drive(final_path)

print(f"\n🎉 All checkpoints are safely backed up in your Google Drive at: {DRIVE_CKPT_DIR}/")
!ls -lh "{DRIVE_CKPT_DIR}"
